# 11 · Poucos dados: com quantos raio-X dá para aprender?

Exercício **"transfer learning com poucas imagens"**: uma MobileNetV2 do ImageNet congelada, uma cabeça
nova, 40 raio-X por classe no treino e 10 na validação, 5 épocas. Lá, numa base de pneumonia, a
acurácia de validação chegou a 95 %. Aqui a mesma ideia vira uma **curva de aprendizado** em
cardiomegalia:

1. **três bases congeladas**: MobileNetV2 e EfficientNetB0 do ImageNet, e uma DenseNet121 treinada em
   raio-X de tórax do **CheXpert** (transferência do mesmo domínio);
2. **a curva**: 10, 20, 40, 100, 300 e todos os 1.775 exames por classe, vários sorteios por tamanho,
   sempre avaliados no **teste inteiro** (1.114 exames), contra a CNN pequena treinada do zero;
3. **quem não precisa de rótulo**: os dois ICTs (perfil, notebook 04; segmentação, notebook 10) só usam
   rótulos para escolher o limiar;
4. **o "95 % com 20 imagens"**: o quanto a validação de 20 exames varia de um sorteio para outro;
5. **o espelhamento**: o exercício usa `horizontal_flip=True`, que põe o coração do lado direito.

A cabeça é uma regressão logística sobre as features (o mesmo que a camada densa sigmoide do
exercício), com a regularização escolhida por validação cruzada **dentro da amostra de treino**. Nenhuma
escolha olha o teste.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import warnings

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.model import avaliacao, cnn, pretreinada, transferencia, unet
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import segmentacao

configurar_estilo()
warnings.filterwarnings("ignore", message=".*normaliz.*")
SECAO = "11-poucos-dados"
TAMANHOS = [10, 20, 40, 100, 300, 1775]
SORTEIOS_CABECA = 10
SORTEIOS_CNN = {10: 5, 20: 5, 40: 5, 100: 5, 300: 3}
# True refaz a curva da CNN (≈ 1 h na CPU); False reaproveita data/processed/curva_cnn.csv.
RETREINAR = False

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
ids = catalogo["id"].to_numpy()
tr, va, te = conjunto == "treino", conjunto == "validacao", conjunto == "teste"
print({c: int((conjunto == c).sum()) for c in ["treino", "validacao", "teste"]})

features = {b: transferencia.extrair_com_cache(b, ids, imagens) for b in transferencia.BASES}
for b, f in features.items():
    print(f"{b}: {f.shape[1]} features por exame")

## 1. Três bases congeladas

Todas recebem o raio-X 128 × 128 ampliado para 224 × 224, e nenhum peso delas muda. As do ImageNet
aprenderam a reconhecer cães, carros e xícaras; a do CheXpert aprendeu 14 achados em 224 mil raio-X de
Stanford, entre eles cardiomegalia. Não há imagem do NIH no CheXpert, então ela nunca viu os nossos
exames.

Antes da curva, o teto: cada cabeça treinada com o treino inteiro (1.775 exames por classe).

In [ ]:
teto, completas = [], {}  # as cabeças do treino inteiro são reaproveitadas nas seções 2 e 4
i_todos = transferencia.sortear(y, tr, 1775, semente=0)
for b, f in features.items():
    modelo = completas[b] = transferencia.cabeca(1775).fit(f[i_todos], y[i_todos])
    teto.append({"base": b, **avaliacao.auc_bootstrap(y[te], modelo.predict_proba(f[te])[:, 1]),
                 "C_escolhido": float(modelo[-1].C_)})
teto = pd.DataFrame(teto).set_index("base")
salvar_tabela(teto, SECAO, "teto-por-base")
teto.round(3)

## 2. A curva de aprendizado

Para cada tamanho, `n` exames por classe sorteados do treino (10 sorteios). As cabeças treinam em
segundos. A **CNN pequena** do notebook 05 treina do zero com o mesmo sorteio e a mesma receita de lá
(aumento sem espelhar, até 150 épocas, paciência 15), com parada antecipada numa validação também
pequena (um quarto de `n` por classe, no mínimo 5), como no exercício. No treino inteiro, o ponto da CNN
é o próprio modelo do notebook 05.

Os dois **ICTs** não aprendem nada dos rótulos: a AUC deles é a mesma com 10 ou 1.775 exames. Entram
como linhas de referência.

In [ ]:
linhas = []
for n in TAMANHOS:
    for s in range(SORTEIOS_CABECA if n < 1775 else 1):
        i = transferencia.sortear(y, tr, n, semente=s)
        for b, f in features.items():
            p = transferencia.cabeca(n).fit(f[i], y[i]).predict_proba(f[te])[:, 1]
            linhas.append({"metodo": b, "n_por_classe": n, "sorteio": s, "auc": roc_auc_score(y[te], p)})
curva_cabecas = pd.DataFrame(linhas)
curva_cabecas.groupby(["metodo", "n_por_classe"])["auc"].mean().unstack().round(3)

In [ ]:
arquivo_cnn = config.DATA_PROCESSED / "curva_cnn.csv"
X = cnn.tensores(imagens)
if RETREINAR or not arquivo_cnn.exists():
    linhas = []
    for n in SORTEIOS_CNN:
        for s in range(SORTEIOS_CNN[n]):
            i = transferencia.sortear(y, tr, n, semente=s)
            v = transferencia.sortear(y, va, max(5, n // 4), semente=s)
            keras.utils.set_random_seed(config.SEMENTE + s)
            modelo = cnn.construir_cnn()
            h = cnn.treinar(modelo, X[i], y[i], X[v], y[v], epocas=150, paciencia=15, lote=min(32, 2 * n))
            linhas.append({"metodo": "CNN pequena · do zero", "n_por_classe": n, "sorteio": s,
                           "auc": roc_auc_score(y[te], cnn.prever(modelo, X[te])), "epocas": len(h.history["loss"])})
            print(f"CNN n={n} sorteio {s}: AUC {linhas[-1]['auc']:.3f}", flush=True)
    pd.DataFrame(linhas).to_csv(arquivo_cnn, sep=";", index=False)
# Treino inteiro: a CNN pequena do notebook 05, treinada com os 1.775 por classe e a validação toda.
p05 = pd.read_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";").set_index("id").loc[ids[te], "CNN pequena · cru"]
curva_cnn = pd.concat([pd.read_csv(arquivo_cnn, sep=";"), pd.DataFrame([{
    "metodo": "CNN pequena · do zero", "n_por_classe": 1775, "sorteio": 0, "auc": roc_auc_score(y[te], p05)}])],
    ignore_index=True)
curva = pd.concat([curva_cabecas, curva_cnn], ignore_index=True)
salvar_tabela(curva, SECAO, "curva-por-sorteio", indice=False)

In [ ]:
# Os dois ICTs: o do perfil (notebook 04) e o da segmentação (notebook 10: coração da U-Net / pulmões da PSPNet).
ict_perfil = pd.read_csv(config.DATA_PROCESSED / "ict.csv", sep=";").set_index("id").loc[ids, "ict"].to_numpy()
prob = pretreinada.segmentar_com_cache(ids, imagens)
pulmoes = (prob[:, 1] > 0.5) | (prob[:, 2] > 0.5)
modelo_unet = keras.models.load_model(config.MODELS / "unet_pspnet.keras",
                                      custom_objects={"perda_bce_dice": unet.perda_bce_dice, "dice_suave": unet.dice_suave})
coracao = [segmentacao.maior_componente(p > 0.5) for p in unet.prever(modelo_unet, X)]
larg = lambda ms: np.array([segmentacao.largura_maxima(m) or np.nan for m in ms], float)
ict_seg = larg(coracao) / larg(pulmoes)
sem_rotulo = {"ICT da segmentação (notebook 10)": ict_seg, "ICT do perfil (notebook 04)": ict_perfil}
auc_sem_rotulo = {k: avaliacao.auc_bootstrap(y[te], v[te])["auc"] for k, v in sem_rotulo.items()}

resumo = curva.groupby(["metodo", "n_por_classe"])["auc"].agg(["mean", "min", "max", "count"]).reset_index()
salvar_tabela(resumo, SECAO, "curva-resumo", indice=False)

CORES = {"MobileNetV2 · ImageNet": config.AMARELO, "EfficientNetB0 · ImageNet": config.VERDE,
         "DenseNet121 · CheXpert": config.CORAL, "CNN pequena · do zero": config.AZUL_COBALTO}
fig, ax = plt.subplots(figsize=(9, 5))
for metodo, g in resumo.groupby("metodo"):
    ax.plot(g["n_por_classe"], g["mean"], "o-", color=CORES[metodo], lw=2, label=metodo)
    ax.fill_between(g["n_por_classe"], g["min"], g["max"], color=CORES[metodo], alpha=0.18)
for (nome, auc), ls in zip(auc_sem_rotulo.items(), ["--", ":"]):
    ax.axhline(auc, color=config.ROXO, ls=ls, lw=1.6, label=f"{nome} · {auc:.3f}")
ax.axvline(40, color="#999", lw=0.8)
ax.text(42, 0.47, "40 por classe\n(o exercício)", fontsize=8, color="#666")
ax.set(xscale="log", xticks=TAMANHOS, xticklabels=TAMANHOS, ylim=(0.45, 0.86),
       xlabel="exames de treino por classe (escala log)", ylabel="AUC no teste (1.114 exames)",
       title="Curva de aprendizado: média e faixa entre sorteios")
ax.legend(fontsize=8, loc="lower right")
salvar_figura(fig, SECAO, "curva-de-aprendizado")
plt.show()
resumo.pivot(index="n_por_classe", columns="metodo", values="mean").round(3)

In [ ]:
# Quantos exames cada aprendiz precisa para alcançar os ICTs, que não usam rótulo nenhum?
for nome, auc in auc_sem_rotulo.items():
    print(f"{nome} (AUC {auc:.3f}):")
    for metodo, g in resumo.groupby("metodo"):
        acima = g.loc[g["mean"] >= auc, "n_por_classe"]
        print(f"   {metodo:28s} → {'a partir de ' + str(acima.min()) + ' por classe' if len(acima) else 'não alcança'}")

In [ ]:
# Com o treino inteiro: a DenseNet do CheXpert congelada contra os melhores do projeto, nos mesmos exames.
p_chex = completas["DenseNet121 · CheXpert"].predict_proba(features["DenseNet121 · CheXpert"][te])[:, 1]
anteriores = pd.read_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";").set_index("id").loc[ids[te]]
rivais = {"CNN pequena · cru (notebook 05)": anteriores["CNN pequena · cru"].to_numpy(),
          "MobileNetV2 · ajuste fino (notebook 05)": anteriores["MobileNetV2 · ajuste fino"].to_numpy(),
          "ICT da segmentação (notebook 10)": ict_seg[te]}
pareada = pd.DataFrame({nome: avaliacao.comparar_auc(y[te], p_chex, e) for nome, e in rivais.items()}).T
salvar_tabela(pareada, SECAO, "densenet-chexpert-pareada")
print(f"DenseNet121 · CheXpert congelada, treino inteiro: AUC {roc_auc_score(y[te], p_chex):.3f}")
pareada.round(3)

## 3. O "95 % com 20 imagens"

O exercício mede acurácia em 20 imagens de validação (10 por classe), uma vez. Aqui a mesma receita
(MobileNetV2 congelada, 40 por classe no treino, 10 por classe na validação, limiar 0,5) é repetida em
**100 sorteios**, e cada modelo também é medido no teste inteiro.

Com 20 imagens, cada acerto vale 5 pontos percentuais: a validação só pode dar 50 %, 55 %, ..., 100 %.

In [ ]:
f = features["MobileNetV2 · ImageNet"]
linhas = []
for s in range(100):
    i = transferencia.sortear(y, tr, 40, semente=1000 + s)
    v = transferencia.sortear(y, va, 10, semente=1000 + s)
    modelo = transferencia.cabeca(40).fit(f[i], y[i])
    linhas.append({"sorteio": s, "acuracia_validacao_20": (modelo.predict(f[v]) == y[v]).mean(),
                   "acuracia_teste_1114": (modelo.predict(f[te]) == y[te]).mean(),
                   "auc_teste": roc_auc_score(y[te], modelo.predict_proba(f[te])[:, 1])})
exercicio = pd.DataFrame(linhas)
salvar_tabela(exercicio, SECAO, "validacao-de-20", indice=False)

ic = binomtest(19, 20).proportion_ci(confidence_level=0.95, method="wilson")
print(f"Uma validação de 19/20 = 95 % tem IC 95 % de {ic.low:.0%} a {ic.high:.0%}")
print(f"Acurácia em 20 de validação: de {exercicio['acuracia_validacao_20'].min():.0%} a "
      f"{exercicio['acuracia_validacao_20'].max():.0%} entre sorteios (desvio {exercicio['acuracia_validacao_20'].std():.1%})")
print(f"Acurácia no teste inteiro:   de {exercicio['acuracia_teste_1114'].min():.0%} a "
      f"{exercicio['acuracia_teste_1114'].max():.0%} (desvio {exercicio['acuracia_teste_1114'].std():.1%})")
r = np.corrcoef(exercicio["acuracia_validacao_20"], exercicio["acuracia_teste_1114"])[0, 1]
print(f"Correlação entre a validação de 20 e o teste: r = {r:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
bins = np.arange(0.325, 1.0, 0.05)
ax1.hist(exercicio["acuracia_validacao_20"], bins=bins, color=config.AMARELO, alpha=0.8, label="validação (20 exames)")
ax1.hist(exercicio["acuracia_teste_1114"], bins=np.arange(0.5, 0.8, 0.01), color=config.AZUL_COBALTO, alpha=0.8,
         label="teste (1.114 exames)")
ax1.set(xlabel="acurácia", ylabel="sorteios", title="A mesma receita, 100 sorteios")
ax1.legend(fontsize=8)
ax2.scatter(exercicio["acuracia_validacao_20"] + np.random.default_rng(0).uniform(-0.012, 0.012, 100),
            exercicio["acuracia_teste_1114"], color=config.AZUL_COBALTO, s=18, alpha=0.7)
ax2.set(xlabel="acurácia na validação de 20", ylabel="acurácia no teste inteiro",
        title=f"A validação de 20 prevê o teste? r = {r:.2f}")
salvar_figura(fig, SECAO, "validacao-de-20")
plt.show()

## 4. O espelhamento

O aumento do exercício inclui `horizontal_flip=True`. Em foto de cachorro isso é inofensivo; em raio-X de
tórax põe o coração do lado direito, uma anatomia que quase não existe (dextrocardia). Com a base
congelada, espelhar equivale a juntar ao treino as features dos raio-X espelhados, com o mesmo rótulo.
Comparação pareada: o mesmo sorteio com e sem os espelhados. Na validação cruzada que escolhe a
regularização, cada raio-X e o seu espelho ficam sempre na mesma dobra: separados, a validação veria
uma quase cópia do treino.

In [ ]:
espelhadas = {b: transferencia.extrair_com_cache(b, ids, imagens, espelhar=True) for b in transferencia.BASES}

fig, eixos = plt.subplots(1, 2, figsize=(6, 3.2))
k = int(np.flatnonzero(te & (y == 1))[0])
for ax, (titulo, im) in zip(eixos, [("original", imagens[k]), ("espelhado", imagens[k][:, ::-1])]):
    ax.imshow(im, cmap="gray"); ax.set_title(titulo, fontsize=9); ax.axis("off")
salvar_figura(fig, SECAO, "espelhado-exemplo")
plt.show()

linhas = []
for n in (40, 300):
    for s in range(SORTEIOS_CABECA):
        i = transferencia.sortear(y, tr, n, semente=s)
        for b, f in features.items():
            sem = transferencia.cabeca(n).fit(f[i], y[i])
            # cada raio-X e o seu espelho no mesmo grupo: a validação cruzada interna nunca os separa
            y_dobro, grupos = np.concatenate([y[i], y[i]]), np.concatenate([i, i])
            com = transferencia.cabeca(n, grupos=grupos, y=y_dobro).fit(np.vstack([f[i], espelhadas[b][i]]), y_dobro)
            linhas.append({"base": b, "n_por_classe": n, "sorteio": s,
                           "auc_sem": roc_auc_score(y[te], sem.predict_proba(f[te])[:, 1]),
                           "auc_com_espelho": roc_auc_score(y[te], com.predict_proba(f[te])[:, 1])})
espelho = pd.DataFrame(linhas).assign(diferenca=lambda d: d["auc_com_espelho"] - d["auc_sem"])
salvar_tabela(espelho, SECAO, "espelhamento", indice=False)
espelho.groupby(["base", "n_por_classe"])[["auc_sem", "auc_com_espelho", "diferenca"]].mean().round(3)

In [ ]:
# A cabeça treinada sem espelho distingue o raio-X espelhado? Se sim, o lado do coração é informação.
linhas = []
for b, f in features.items():
    p_orig = completas[b].predict_proba(f[te])[:, 1]
    p_esp = completas[b].predict_proba(espelhadas[b][te])[:, 1]
    linhas.append({"base": b, "auc_original": roc_auc_score(y[te], p_orig), "auc_no_espelhado": roc_auc_score(y[te], p_esp)})
pd.DataFrame(linhas).set_index("base").round(3)

## Conclusões

* **A base do mesmo domínio ganha em todos os tamanhos.** A DenseNet121 do CheXpert congelada, só com
  uma regressão logística em cima, dá AUC **0,705** com 40 exames por classe (o tamanho do exercício),
  contra 0,63 das duas bases ImageNet. Com 10 por classe, 0,60 contra 0,56. Entre as do ImageNet, a
  EfficientNetB0 não ganha da MobileNetV2 até o treino inteiro (0,784 × 0,774).
* **Com poucos dados, a CNN do zero não aprende.** Até 100 por classe ela fica em 0,50 a 0,56, perto do
  acaso, enquanto as bases congeladas já passam de 0,67. Ela só alcança as outras com o treino inteiro
  (0,802, o modelo do notebook 05). O transfer learning troca rótulos por conhecimento prévio.
* **Medir ainda vale mais que aprender com poucos rótulos.** O ICT da segmentação (notebook 10) dá
  AUC 0,803 sem usar rótulo nenhum de cardiomegalia. Nenhum aprendiz chega lá antes do treino inteiro,
  e o ICT do perfil (0,739) só é alcançado pela DenseNet do CheXpert a partir de 100 por classe.
* **O melhor modelo do projeto é o mais simples de treinar.** Com os 1.775 por classe, a DenseNet do
  CheXpert congelada chega a **AUC 0,826**: +0,031 sobre a MobileNetV2 com ajuste fino (IC +0,005 a
  +0,057), +0,023 sobre a CNN pequena (IC −0,003 a +0,048) e +0,024 sobre o ICT da segmentação
  (IC −0,009 a +0,057). Só a primeira diferença é clara. A ressalva: o CheXpert tem rótulo de
  cardiomegalia, então a base já aprendeu o conceito em outro hospital. É transferência legítima (sem
  imagem do NIH), e é por isso que ela vai tão bem.
* **"95 % com 20 imagens" não mede o modelo.** Com a mesma receita do exercício em 100 sorteios, a
  acurácia na validação de 20 vai de **25 % a 95 %**, enquanto no teste inteiro fica entre 52 % e 65 %.
  As duas quase não se correlacionam (r = 0,16). Um único 19/20 tem IC 95 % de 76 % a 99 %. Com tão
  pouca validação, o número final é mais sorte do sorteio que qualidade do modelo, e cardiomegalia é
  bem mais difícil que pneumonia.
* **Espelhar só atrapalha a base que conhece anatomia.** Juntar os raio-X espelhados ao treino tira
  0,010 de AUC da DenseNet do CheXpert com 40 e com 300 por classe (pior em 7 e 8 de 10 sorteios). Nas
  bases ImageNet o efeito é pequeno e até positivo (+0,003 a +0,010): elas nem distinguem o espelho
  (AUC 0,774 → 0,781 no teste espelhado), porque foram treinadas para ignorá-lo. A do CheXpert cai de
  0,826 para **0,685** no teste espelhado: ela aprendeu que o coração fica à esquerda, e é esse
  conhecimento que o espelhamento atrapalha. Na validação cruzada, cada raio-X e o seu espelho ficam
  na mesma dobra; separados, a validação via uma quase cópia do treino e o espelhamento parecia piorar
  as três bases.